# Thermal Noise

This notebook promotes the noise-floor and sensitivity ideas that were implicit in the legacy AM/FM demo into a dedicated lesson. It connects the `kTB` equation to practical receiver bandwidth and noise figure.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## Room-Temperature Noise Power

At 290 K, thermal noise density is approximately `-174 dBm/Hz`. Over a receiver bandwidth `B`, the noise floor is:

$$P_n = -174\,\text{dBm/Hz} + 10\log_{10}(B) + NF$$

where `NF` is the receiver noise figure in dB.

In [ ]:
def thermal_noise_dbm(bandwidth_hz, noise_figure_db=0.0):
    return -174 + 10 * np.log10(bandwidth_hz) + noise_figure_db

bandwidths = np.array([2500, 6250, 12_500, 25_000, 200_000])
for bw in bandwidths:
    print(f"{bw:>7,d} Hz -> {thermal_noise_dbm(bw, noise_figure_db=6):6.2f} dBm with 6 dB NF")


In [ ]:
out = widgets.Output()

def update_noise_floor(bandwidth_hz=12_500.0, noise_figure_db=6.0, required_sinad=12.0):
    with out:
        out.clear_output(wait=True)
        noise_floor = thermal_noise_dbm(bandwidth_hz, noise_figure_db)
        sensitivity = noise_floor + required_sinad
        print(f"Noise floor: {noise_floor:.2f} dBm")
        print(f"Estimated sensitivity for {required_sinad:.1f} dB SINAD: {sensitivity:.2f} dBm")

controls = widgets.interactive(
    update_noise_floor,
    bandwidth_hz=float_slider(min_value=2500, max_value=200000, step=2500, value=12500, description="BW Hz", readout_format=".0f"),
    noise_figure_db=float_slider(min_value=0, max_value=15, step=0.5, value=6, description="NF dB"),
    required_sinad=float_slider(min_value=3, max_value=20, step=0.5, value=12, description="SINAD"),
)
display(controls, out)


## From Equations to Samples

We can simulate the effect of thermal-like white noise by adding AWGN to a tone and watching the spectrum floor rise as SNR drops.

In [ ]:
fs = 48_000
t, tone = generate_tone(freq=1000, duration=0.2, fs=fs, amplitude=1.0)

fig, ax = plt.subplots(figsize=(10, 3.5))

def update_awgn(snr_db=20.0):
    ax.clear()
    noisy, _ = add_awgn(tone, snr_db=snr_db, seed=42)
    plot_spectrum(noisy, fs=fs, ax=ax, title=f"Tone + AWGN, SNR={snr_db:.0f} dB")
    ax.set_xlim(0, 4000)
    ax.set_ylim(-120, 5)
    fig.canvas.draw_idle()

controls = widgets.interactive(
    update_awgn,
    snr_db=float_slider(min_value=0, max_value=40, step=1, value=20, description="SNR dB"),
)
display(controls)


## Key Takeaway

Thermal noise is not a special pathology; it is the baseline. Receiver sensitivity starts with `kTB`, then every extra bandwidth and every dB of noise figure makes the floor worse.